<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 1</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">1.4</span>Optimización de portafolios que sí funciona</div>
</div>

**Objetivo.** ver por qué la optimización clásica falla fuera de muestra y probar tres alternativas más robustas.

**Lo que vamos a hacer**

- Estimar la covarianza con y sin *shrinkage* (Ledoit–Wolf).
- Construir portafolios: mínima varianza, HRP y mínimo CVaR.
- Evaluar todos fuera de muestra contra el portafolio igual-ponderado.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Datos y partición

In [ ]:
tickers = ["EC", "CIB", "AAPL", "MSFT", "JPM", "XOM", "GLD", "TLT"]
prices = download_prices(tickers, start="2016-01-01")
rets = to_returns(prices, log=False)
n = rets.shape[1]
split = int(0.7 * len(rets))
train, test = rets.iloc[:split], rets.iloc[split:]
print(f"Entrenamiento: {len(train)} días · Prueba: {len(test)} días · {n} activos")

## 2. Covarianza con shrinkage

La covarianza muestral es ruidosa: el optimizador explota ese ruido. **Ledoit–Wolf** la mezcla con una matriz simple para estabilizarla.

> ### ✏️ Ejercicio 1 — Ledoit–Wolf
> Estima `cov_lw` (anualizada) con `sklearn.covariance.LedoitWolf` y muestra cuánto *shrinkage* aplicó.

In [ ]:
from sklearn.covariance import LedoitWolf
cov_s = train.cov().values * 252
# TODO: cov_lw = ...
cov_lw = cov_s

In [ ]:
print(f"Número de condición — muestral: {np.linalg.cond(cov_s):,.0f} | Ledoit–Wolf: {np.linalg.cond(cov_lw):,.0f}")

from scipy.optimize import minimize, linprog
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

def min_var(cov):
    n = len(cov)
    res = minimize(lambda w: w @ cov @ w, np.ones(n) / n, bounds=[(0, 1)] * n,
                   constraints={"type": "eq", "fun": lambda w: w.sum() - 1}, method="SLSQP")
    return res.x

def _cluster_var(cov, items):
    sub = cov[np.ix_(items, items)]
    ivp = 1 / np.diag(sub); ivp /= ivp.sum()
    return ivp @ sub @ ivp

def hrp(cov):
    """Hierarchical Risk Parity (López de Prado, 2016)."""
    cov = np.asarray(cov)
    sd = np.sqrt(np.diag(cov)); corr = cov / np.outer(sd, sd)
    dist = np.sqrt(np.clip((1 - corr) / 2, 0, None))
    order = list(leaves_list(linkage(squareform(dist, checks=False), "single")))
    w = pd.Series(1.0, index=order)
    clusters = [order]
    while clusters:
        clusters = [c[j:k] for c in clusters for j, k in ((0, len(c) // 2), (len(c) // 2, len(c))) if len(c) > 1]
        for a in range(0, len(clusters), 2):
            c0, c1 = clusters[a], clusters[a + 1]
            v0, v1 = _cluster_var(cov, c0), _cluster_var(cov, c1)
            alpha = 1 - v0 / (v0 + v1)
            w[c0] *= alpha; w[c1] *= 1 - alpha
    return w.sort_index().values

def min_cvar(R, beta=0.95):
    """Min-CVaR (Rockafellar–Uryasev) como programa lineal. R: matriz T×n de retornos."""
    T, n = R.shape
    c = np.r_[np.zeros(n), 1.0, np.ones(T) / ((1 - beta) * T)]          # variables: [w, alpha, u]
    A_ub = np.hstack([-R, -np.ones((T, 1)), -np.eye(T)])               # u_t >= -R_t·w - alpha
    A_eq = np.r_[np.ones(n), 0.0, np.zeros(T)][None, :]
    bounds = [(0, 1)] * n + [(None, None)] + [(0, None)] * T
    res = linprog(c, A_ub=A_ub, b_ub=np.zeros(T), A_eq=A_eq, b_eq=[1], bounds=bounds, method="highs")
    return res.x[:n]

## 3. Construir los portafolios

> ### ✏️ Ejercicio 2 — Diccionario de pesos
> Crea `W` (DataFrame) con las columnas: `Igual`, `MinVar muestral`, `MinVar LW`, `HRP` y `Min CVaR` (usa `train.values` para CVaR).

In [ ]:
# TODO: W = pd.DataFrame({...}, index=train.columns)
W = None

In [ ]:
W.plot.bar(figsize=(11, 4.5), width=.85, title="Pesos por método")
plt.xticks(rotation=0); plt.legend(ncol=5, loc="upper center", bbox_to_anchor=(.5, -.1)); plt.show()

## 4. Evaluación fuera de muestra

> ### ✏️ Ejercicio 3 — Métricas en la muestra de prueba
> Para cada columna de `W` calcula en `test`: volatilidad anual, Sharpe (rf = 0) y ES 95% diario. Guarda la tabla en `oos`.

In [ ]:
# TODO: oos = pd.DataFrame(...)
oos = None

## Para llevarse

- Un optimizador con covarianza ruidosa «sobreajusta»: pesos extremos que no se sostienen fuera de muestra.
- Shrinkage, HRP y CVaR reducen esa fragilidad por caminos distintos.
- Compara siempre contra el portafolio igual-ponderado: es difícil de superar.

---
*Excel para mirar, Python para repetir.*